## Web Intelligence - 2024
#### Assignment 3 - Part I

In this assignment, you will work with the *IMDB movie review dataset* (https://huggingface.co/datasets/stanfordnlp/imdb), which contains $25,000$ highly polarized movie reviews for training and another $25,000$ for testing. Our goal is to experiment with binary text classification using pre-learned embeddings trained on the Google News dataset (https://code.google.com/archive/p/word2vec/). We aim to gain deeper insights into neural network architectures by implementing various models for this task.

Let's begin by importing the necessary packages.

In [1]:
# Tested for Python=3.9
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from torch.optim import Adam
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from datasets import load_dataset # 
import gensim

#### Loading and processing the IMDB movie review dataset

We will use the ```datasets``` package developed by the HuggingFace team (https://github.com/huggingface/datasets). The '**IMDB**' dataset can be loaded with the ```load_dataset``` method.

In [2]:
imdb = load_dataset("imdb")
reviews_train = imdb['train']["text"]
reviews_test = imdb['test']["text"]

We construct a dictionary for the reviews by ```reviews_train```, including only terms that have a minimum frequency of 0.0005 and a maximum frequency of 0.5.

- Analyze how changing these parameters affects your model’s accuracy.

In [3]:
# Convert a collection of text documents to a sparse matrix of token counts.
# min_df and max_df: When building the vocabulary ignore terms that have a document frequency strictly lower and greater  
# than the given threshold. This value is also called cut-off in the literature
dictionary = CountVectorizer(min_df=0.0005, max_df=0.5).fit(reviews_train)

# Sparse matrix representation
reviews_train_dict = dictionary.transform(reviews_train)
reviews_test_dict = dictionary.transform(reviews_test)

# Get the document(i.e. review)-term matrix
reviews_train_dt = torch.from_numpy(reviews_train_dict.todense())
reviews_test_dt = torch.from_numpy(reviews_test_dict.todense())
print(f"Shape of reviews_train:{reviews_train_dt.shape}")

Shape of reviews_train:torch.Size([25000, 15862])


#### Movie Review Representations

To use the reviews as input for our neural network architecture, we need to represent them as vectors. For this purpose, we’ll use pre-trained word embeddings (https://code.google.com/archive/p/word2vec/) to construct vector representations of the reviews in a latent space.

- Please note that loading these embeddings may take some time, so a little patience is recommended! :)

In [4]:
model = gensim.models.KeyedVectors.load_word2vec_format('./GoogleNews-vectors-negative300.bin.gz', binary = True)

We define the review embedding vectors, $E[r]$, as the weighted sum of word vectors with respect to their frequencies. In other words, 

$$ E[r] = \sum_{w \ \in \ r} \frac{counts[w, r]}{|r|}  \cdot  E_{w}[w]$$

where $E_{w}[w]$ is the pre-trained 'word' embedding, $counts[w, r]$ indicates the number of appearances of word, 'w', in a given review, 'r', and $|r|$ shows the number words in review, 'r'.

The word embedding matrix has dimensions of #words_in_dictionary x #embedding_dim, and it is constructed by iterating over each word in the dictionary created above. If a word does not have a corresponding embedding (i.e., it is not found in the embedding model), it is assigned a zero vector.

In [5]:
word_embs_matrix = torch.zeros((len(dictionary.vocabulary_), 300), dtype=torch.float)

words_not_found = []
for k, i in dictionary.vocabulary_.items():
    try:
        word_embs_matrix[i, :] = torch.tensor(model[k])
    except KeyError:
        words_not_found.append(k)
        
print(f"Number of failed look up attempts for the dictionary: {len(words_not_found)}")
print(f"Examples include: {words_not_found[:10]}")

reviews_train_df_normalized = reviews_train_dt / reviews_train_dt.sum(axis=1, keepdim=True)
reviews_test_df_normalized= reviews_test_dt / reviews_test_dt.sum(axis=1, keepdim=True)
review_train_embs = reviews_train_df_normalized @ word_embs_matrix
review_test_embs = reviews_test_df_normalized @ word_embs_matrix
print(f"The resulting matrix (review_train_embs) shape: {review_train_embs.shape}")

Number of failed look up attempts for the dictionary: 1145
Examples include: ['1967', '40', 'ingmar', 'bergman', '17', 'godard', '10', 'lucille', 'ritter', 'bogdanovich']
The resulting matrix (review_train_embs) shape: torch.Size([25000, 300])



#### Artificial Neural Network Design

Below is a simple example of how to set up a neural network using PyTorch to experiment with classifying movie reviews based on their embedding representations. In this task, you should:

- Experiment with various neural network architectures and learning configurations (i.e. batch size, learning rate, number of epochs) to assess their impact on classification accuracy. Investigate how performance is influenced by factors such as the number of hidden layers, the number of neurons, and the choice of activation functions. To support this analysis, you can define a validation dataset to be used during model training. 

- Adjust the cutoff frequencies when constructing the IMDB dictionary. Observe how changing these parameters affects the accuracy of your model.

In the example code provided below, we use the ```torch.nn module``` from PyTorch. For an introduction to this module and instructions on how to define neural networks in PyTorch, refer to the following tutorials:

- https://pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html#sphx-glr-beginner-blitz-neural-networks-tutorial-py
- https://pytorch.org/tutorials/beginner/nn_tutorial.html

We can set the platform that we would like to use to train the architecture, and a seed value for reproducibility.

In [6]:
device = "cuda"  if torch.cuda.is_available() else "cpu"
seed = 123456789
print(f"Using {device} device")

torch.manual_seed(seed)

Using cpu device


First, we define a dataloader to handle the dataset, and we will use mini-batches, which form the basis for the stochastic gradient descent implementation.

In [7]:
batch_size = 64 # Set the batch size
epochs = 10 # Set the number of epochs
lr = 1e-3 # Set the learning rate

train = torch.utils.data.TensorDataset(review_train_embs, torch.tensor(imdb['train']['label']))
test = torch.utils.data.TensorDataset(review_test_embs, torch.tensor(imdb['test']['label']))

train_loader = torch.utils.data.DataLoader(
    train, batch_size=batch_size, shuffle=True, num_workers=0
)
test_loader = torch.utils.data.DataLoader(
    test, batch_size=1, shuffle=False, drop_last=False, num_workers=0
)

Here, we will define our artificial neural network.

In [8]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Implement the constructor 
        self.linear_relu_stack = torch.nn.Sequential(
            torch.nn.Linear(300, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 2),
        )
        ##############################

    def forward(self, x):
        # Impement the forward method
        logits = self.linear_relu_stack(x)
        return logits
        ##############################

In [9]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=300, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=2, bias=True)
  )
)


Define the training and testing functions.

In [10]:
# Adapted from https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    train_loss = 0
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)
        train_loss += loss.item()
        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    return train_loss / len(dataloader)

def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return test_loss, correct


Finally, let's train the model and see how it performs.

In [11]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.5**(2/epochs))

test_loop(test_loader, model, loss_fn)

pbar = tqdm(range(epochs))
for epoch in pbar:
    train_loss = train_loop(train_loader, model, loss_fn, optimizer)
    test_loss, accuracy = test_loop(test_loader, model, loss_fn)
    scheduler.step()
    pbar.set_description(f"Train_loss: {train_loss:.2f}, Test_loss: {test_loss:.2f}, accuracy: {100*accuracy:0.2f}")

test_loop(test_loader, model, loss_fn)
print("Done!")

Test Error: 
 Accuracy: 50.0%, Avg loss: 0.694078 



  0%|          | 0/10 [00:00<?, ?it/s]

Test Error: 
 Accuracy: 83.4%, Avg loss: 0.378844 

Test Error: 
 Accuracy: 84.8%, Avg loss: 0.353875 

Test Error: 
 Accuracy: 84.7%, Avg loss: 0.352790 

Test Error: 
 Accuracy: 84.4%, Avg loss: 0.357948 

Test Error: 
 Accuracy: 85.6%, Avg loss: 0.338522 

Test Error: 
 Accuracy: 85.2%, Avg loss: 0.342852 

Test Error: 
 Accuracy: 85.5%, Avg loss: 0.339640 

Test Error: 
 Accuracy: 85.9%, Avg loss: 0.333779 

Test Error: 
 Accuracy: 86.0%, Avg loss: 0.332974 

Test Error: 
 Accuracy: 85.6%, Avg loss: 0.334098 

Test Error: 
 Accuracy: 85.6%, Avg loss: 0.334098 

Done!
